In [4]:
#
import pandas as pd
import multiprocess as mp 
import warnings
warnings.filterwarnings("ignore")

file_names = {
            "cleafshrub": 'obsfile_Shrub_cLeaf_y.txt',
              "cstemshrub": 'obsfile_Shrub_cStem_y.txt',
              "anppshrub": 'obsfile_Shrub_ANPP_y.txt',
              "bnppshrub_d": 'obsfile_Shrub_BNPP_d.txt',
              "laishrub": 'obsfile_Shrub_LAI_h.txt',
              "cplantsphag": 'obsfile_Sphag_cPlant_y.txt',
              "nppsphag": 'obsfile_Sphag_NPP_y.txt',
              
              "cplanttree": 'obsfile_Tree_PlantC_y.txt',
              "laitree": 'obsfile_Tree_LAI_h.txt',
              "phototree": 'obsfile_Tree_Photo_h.txt',
              "anpptree": 'obsfile_Tree_ANPP_y.txt',
              "bnpptree_d": 'obsfile_Tree_BNPP_d.txt',
              
              "gpp": 'obsfile_gpp_h.txt',
              "er": 'obsfile_er_h.txt',
              "nee": 'obsfile_nee_h.txt',
              "rh": 'obsfile_rh_h.txt',
              "ch4": 'obsfile_ch4_h.txt',
              "csoil": 'obsfile_cSoil_y.txt',}

# modPath = "../1_simulations/16_final_simu_daynpp_3/outputs/"
# obsPath = "../1_simulations/16_final_simu_daynpp_3/TECO-SPRUCE_v3.1/inputs/in_alltreat/"
modPath = "../../data_results/1_data_source/1_hourly_simulations/"
obsPath = "../../data_results/1_data_source/1_in_alltreat/"


plot_names = ["P04", "P06",  "P08", "P10", "P11", "P13", "P16", "P17", "P19", "P20"]
# iplot = "P04"
comName = "run_mcmc_alltreat_"

def run_plot(iplot):
    print(iplot)
    df_mod = pd.read_csv(modPath+f"TECO-SPRUCE_{comName}{iplot}_Hourly.csv")
    df_mod = df_mod.rename(columns={' year': 'year'})
    for ivar, ifile in file_names.items():
        # print(ivar)
        df_obs = pd.read_csv(obsPath+f"{iplot}/observation_files/"+ifile)
        if ivar == "bnpp": # yearly
            df_sel_mod = df_mod[["year", "doy", "hour", "nppRoot_Tree", "nppRoot_Shrub"]].copy()
            df_sel_mod["simu"] = 0.5 * df_mod["nppRoot_Tree"] + 0.25 * df_mod["nppRoot_Shrub"]
            df_sel_mod = df_sel_mod[["year", "doy", "hour", "simu"]]
            df_sel_mod_yr = df_sel_mod.groupby(["year"], as_index=False)["simu"].sum()
            df_obs = df_obs.merge(df_sel_mod_yr, on="year", how="left")
        elif ivar == "anppshrub": 
            df_sel_mod = df_mod[["year", "doy", "hour", "nppLeaf_Shrub", "nppStem_Shrub"]]
            df_sel_mod["simu"] = 0.25 * df_mod["nppLeaf_Shrub"] + 0.25 * df_mod["nppStem_Shrub"]
            df_sel_mod_yr = df_sel_mod.groupby(["year"], as_index=False)["simu"].sum()
            df_obs = df_obs.merge(df_sel_mod_yr, on="year", how="left")
        elif ivar == "bnppshrub":
            df_sel_mod = df_mod[["year", "doy", "hour", "nppRoot_Shrub"]].copy()
            df_sel_mod["simu"] = 0.25 * df_mod["nppRoot_Shrub"]
            df_sel_mod_yr = df_sel_mod.groupby(["year"], as_index=False)["simu"].sum()
            df_obs = df_obs.merge(df_sel_mod_yr, on="year", how="left")
        elif ivar == "photoshrub": 
            df_sel_mod = df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = df_mod["Aleaf_sum_Shrub"]
            df_obs = df_obs.merge(df_sel_mod[["year", "doy", "hour","simu"]], on=["year", "doy", "hour"], how="left")
        elif ivar == "cleafshrub": 
            df_sel_mod = df_mod[["year", "doy", "hour", "cLeaf_Shrub"]].copy()
            df_sel_mod["simu"] = 0.25 * df_mod["cLeaf_Shrub"]
            df_obs = df_obs.merge(df_sel_mod[["year", "doy", "hour","simu"]], on=["year", "doy", "hour"], how="left")
        elif ivar == "cstemshrub": 
            df_sel_mod = df_mod[["year", "doy", "hour", "cStem_Shrub"]].copy()
            df_sel_mod["simu"] = 0.25 * df_mod["cStem_Shrub"]
            df_obs = df_obs.merge(df_sel_mod[["year", "doy", "hour","simu"]], on=["year", "doy", "hour"], how="left") 
        elif ivar == "nppsphag":
            df_sel_mod = df_mod[["year", "doy", "hour", "npp_Sphagnum"]].copy()
            df_sel_mod["simu"] = 0.25 * df_mod["npp_Sphagnum"]
            df_sel_mod_yr = df_sel_mod.groupby(["year"], as_index=False)["simu"].sum()
            df_obs = df_obs.merge(df_sel_mod_yr, on="year", how="left")
        elif ivar == "cplantsphag":
            df_sel_mod = df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = 0.25 * (df_mod["cLeaf_Sphagnum"] + df_mod["cStem_Sphagnum"] + df_mod["cRoot_Sphagnum"])
            df_obs = df_obs.merge(df_sel_mod[["year", "doy", "hour","simu"]], on=["year", "doy", "hour"], how="left") 
        elif ivar == "anpptree":
            df_sel_mod = df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = 0.5 * (df_mod["nppLeaf_Tree"] + df_mod["nppStem_Tree"])
            df_sel_mod_yr = df_sel_mod.groupby(["year"], as_index=False)["simu"].sum()
            df_obs = df_obs.merge(df_sel_mod_yr, on="year", how="left")
        elif ivar == "bnpptree":
            df_sel_mod = df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = 0.5 * (df_mod["nppRoot_Tree"])
            df_sel_mod_yr = df_sel_mod.groupby(["year"], as_index=False)["simu"].sum()
            df_obs = df_obs.merge(df_sel_mod_yr, on="year", how="left")
        elif ivar == "laitree":
            df_sel_mod = df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = df_mod["lai_Tree"]
            df_obs = df_obs.merge(df_sel_mod, on=["year", "doy", "hour"], how="left")
        elif ivar == "phototree":
            df_sel_mod = df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = (df_mod["Aleaf_sum_Tree"])
            df_obs = df_obs.merge(df_sel_mod, on=["year", "doy", "hour"], how="left")
        elif ivar == "cplanttree": 
            df_sel_mod = df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = 0.5 * (df_mod["cLeaf_Tree"] + df_mod["cStem_Tree"] )
            df_obs = df_obs.merge(df_sel_mod, on=["year", "doy", "hour"], how="left")
        elif ivar == "csoil": 
            df_sel_mod = df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = df_mod["cSoil"]
            df_obs = df_obs.merge(df_sel_mod, on=["year", "doy", "hour"], how="left")
        elif ivar == "ch4": 
            df_sel_mod = df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = df_mod["wetlandCH4"]
            df_obs = df_obs.merge(df_sel_mod, on=["year", "doy", "hour"], how="left")
        elif ivar == "er":
            df_sel_mod =  df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = 0.25 * df_mod["ra_Shrub"] + 0.25 * df_mod["ra_Sphagnum"] + df_mod["rh"] 
            # df_sel_mod["simu"] = 0.25 * df_mod["npp_Shrub"] + 0.25 * df_mod["npp_Sphagnum"] #+ df_mod["rh"] 
            df_obs = df_obs.merge(df_sel_mod, on=["year", "doy", "hour"], how="left")
        elif ivar == "gpp":
            df_sel_mod =  df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = 0.25 * df_mod["gpp_Shrub"] + 0.25 * df_mod["gpp_Sphagnum"]
            df_obs = df_obs.merge(df_sel_mod, on=["year", "doy", "hour"], how="left")
        elif ivar == "nee":
            df_sel_mod =  df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = 0.25 * df_mod["ra_Shrub"] + 0.25 * df_mod["ra_Sphagnum"] + df_mod["rh"] - (0.25 * df_mod["gpp_Shrub"] + 0.25 * df_mod["gpp_Sphagnum"])
            df_obs = df_obs.merge(df_sel_mod, on=["year", "doy", "hour"], how="left")
        elif ivar == "rh":
            df_sel_mod =  df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = df_mod["rh"] 
            df_obs = df_obs.merge(df_sel_mod, on=["year", "doy", "hour"], how="left")
        elif ivar == "laishrub":
            df_sel_mod = df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = (df_mod["lai_Shrub"])
            df_obs = df_obs.merge(df_sel_mod, on=["year", "doy", "hour"], how="left")
        elif ivar == "bnppshrub_d":
            df_sel_mod = df_mod[["year", "doy", "hour", "nppRoot_Shrub"]].copy()
            df_sel_mod["simu"] = 0.25 * df_mod["nppRoot_Shrub"]
            df_sel_mod_d = df_sel_mod.groupby(["year", "doy"], as_index=False)["simu"].sum()
            df_obs = df_obs.merge(df_sel_mod_d, on=["year", "doy"], how="left")
        elif ivar == "bnpptree_d":
            df_sel_mod = df_mod[["year", "doy", "hour"]].copy()
            df_sel_mod["simu"] = 0.5 * (df_mod["nppRoot_Tree"])
            df_sel_mod_d = df_sel_mod.groupby(["year", "doy"], as_index=False)["simu"].sum()
            df_obs = df_obs.merge(df_sel_mod_d, on=["year", "doy"], how="left")
        df_obs.to_excel(f"../../data_results/2_results/2-2_obs_vs_simu/{iplot}_{ivar}.xlsx")

# run_plot("P11")
# plot_names = ["P08", "P10", "P11", "P13", "P16", "P17", "P19", "P20"]
# plot_names = ["P06","P08","P10", "P20"]
with mp.Pool(10) as pool:
    pool.map(run_plot, plot_names)

P04P06P08P10P11P13P16P19P17P20









